<a href="https://colab.research.google.com/github/atenine/LING-230-Single-Model-Disfluency-Detector/blob/optimization/scripts/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from huggingface_hub import login
from dotenv import dotenv_values
import pandas as pd
from transformers import AutoConfig, AutoModel
import torchaudio
import torch
import torch.nn as nn
import numpy as np
from datetime import datetime
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler

%load_ext tensorboard

login(dotenv_values()['HUGGINGFACE_API_KEY'])



In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
torch.cuda.empty_cache()
import gc
gc.collect()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install gdown

import os
import tarfile

# Ensure /content/data directory exists for extraction
os.makedirs('/content/data', exist_ok=True)

# Google Drive file ID for the zip file provided by the user
file_id = '18I3VYO_Le-_cFY6Rmbg1MLbHrwmwPE6F'
zip_file_name = '/content/data/SEP-28k_CLIP.zip'

print(f"Downloading {zip_file_name} from Google Drive...")
# Download the file from Google Drive
!gdown --id "$file_id" -O "$zip_file_name"

print(f"Extracting {zip_file_name}...")
# Extract the zip file to /content/data/
with tarfile.open(zip_file_name, 'r') as tar:
    tar.extractall('/content/data/')

# Define the paths based on the extracted data
clips_path = '/content/data/SEP-28k_CLIP'


print("Data setup complete.")
print(f"Clips path set to: {clips_path}")
print("Clips directory exists:", os.path.exists(clips_path))

In [ ]:
csv_path   = '/content/SEP-28k-Extended_clips.csv'
clips_path = '/content/data/SEP-28k_CLIP'
print("csv exist:", os.path.exists(csv_path))
print("Clips exist:", os.path.exists(clips_path))

In [ ]:
# Check CSV
print("CSV exists:", os.path.exists(csv_path))

# Check one audio clip
path = clips_path + '/HeStutters/10/HeStutters_10_0.wav'
print("Data Exists:", os.path.exists(path))


In [ ]:
# Load CSV
df = pd.read_csv(csv_path)

# Build filepath — Show/EpId/Show_EpId_ClipId.wav
df['filepath'] = df.apply(
    lambda row: os.path.join(
        clips_path,
        str(row['Show']),
        str(row['EpId']),
        f"{row['Show']}_{row['EpId']}_{row['ClipId']}.wav"
    ), axis=1
)

# Check how many files exist
df['exists'] = df['filepath'].apply(os.path.exists)
print(f"Total clips in CSV:     {len(df)}")
print(f"Clips found on disk:    {df['exists'].sum()}")
print(f"Clips NOT found:        {(~df['exists']).sum()}")

# Keep only existing files
df = df[df['exists']]
print("\nSample filepath:")
print(df['filepath'].iloc[0])

In [ ]:
stutter_cols = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection']
label_map={
    'Prolongation': 'prolongation',
    'Block': 'block',
    'SoundRep': 'soundrep',
    'WordRep': 'wordrep',
    'Interjection': 'interjection',
    'NoStutteredWords':'fluent'
}
cols_to_normalize = list(label_map.keys())
df[list(label_map.values())] = df[cols_to_normalize].div(3.0).rename(columns=label_map)

print("\nAverage probability per stutter type:")
for new_col in list(label_map.values())[:-1]:
    print(f"{new_col}: {df[new_col].mean():.3f}")

In [ ]:
subset_fraction = 0.2
train_df = df[df['SEP28k-E']== 'train'].sample(frac=subset_fraction, random_state=42).reset_index(drop=True)
valid_df = df[df['SEP28k-E']== 'dev'].sample(frac=subset_fraction, random_state=42).reset_index(drop=True)
test_df = df[df['SEP28k-E']== 'test'].sample(frac=subset_fraction, random_state=42).reset_index(drop=True)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# # load voc2vec Encoder
# print("\n Loading voc2vec encoder...")
# # MODEL_NAME = "alkiskoudounas/voc2vec-hubert-ls-pt" # facebook/wav2vec2-base, alkiskoudounas/voc2vec-hubert-ls-pt, openai/whisper-large-v3-turbo
# MODEL_NAME = "facebook/wav2vec2-base"
# # MODEL_NAME = "openai/whisper-large-v3-turbo"
# encoder_config = AutoConfig.from_pretrained(MODEL_NAME)
# encoder = AutoModel.from_pretrained(MODEL_NAME, device_map="cuda")

In [ ]:
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3-turbo")

In [ ]:
import soundfile as sf
import numpy as np

# Hyperparameters
batch_size    = 256
num_epochs    = 50
learning_rate = 0.0001

# Paths
SAVE_PATH  = '/content/drive/MyDrive/stutter_embeddings/'
LOCAL_PATH = '/content/embeddings/'
os.makedirs(SAVE_PATH,  exist_ok=True)
os.makedirs(LOCAL_PATH, exist_ok=True)

# Encoders
ENCODERS = {
    'wav2vec': 'facebook/wav2vec2-base',
    'voc2vec': 'alkiskoudounas/voc2vec-hubert-ls-pt',
    'whisper': 'openai/whisper-large-v3-turbo'
}

def filter_valid(df):
    valid = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Filtering"):
        try:
            if sf.info(row['filepath']).frames == 48000:
                valid.append(row)
        except:
            pass
    return pd.DataFrame(valid).reset_index(drop=True)

print("Filtering train...")
train_df_valid = filter_valid(train_df)
print("Filtering val...")
val_df_valid   = filter_valid(valid_df)
print("Filtering test...")
test_df_valid  = filter_valid(test_df)

print(f"Train valid: {len(train_df_valid)}")
print(f"Val valid:   {len(val_df_valid)}")
print(f"Test valid:  {len(test_df_valid)}")

In [ ]:
class StutterDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, is_whisper=False):
        self.df         = dataframe.reset_index(drop=True)
        self.label_cols = ['prolongation', 'block', 'soundrep', 'wordrep', 'interjection']
        self.labels     = torch.tensor(self.df[self.label_cols].values, dtype=torch.float32)
        self.is_whisper = is_whisper

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        waveform, sample_rate = torchaudio.load(row['filepath'])

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sample_rate != 16000:
            waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)

        if self.is_whisper:
            # Whisper needs mel spectrogram
            inputs = whisper_processor(
                waveform.squeeze().numpy(),
                sampling_rate=16000,
                return_tensors='pt'
            )
            return inputs['input_features'].squeeze(0), self.labels[idx], \
                   f"{row['Show']}_{row['EpId']}_{row['ClipId']}"
        else:
            name = f"{row['Show']}_{row['EpId']}_{row['ClipId']}"
            return waveform, self.labels[idx], name


def make_raw_loader(df, is_whisper=False):
    return DataLoader(
        StutterDataset(df, is_whisper=is_whisper),
        batch_size=16,
        num_workers=0,
        pin_memory=False,
        shuffle=False
    )

In [ ]:
def precompute_embeddings(loader, split_name):
    save_file = SAVE_PATH + f'{split_name}_embeddings.npy'
    if os.path.exists(save_file):
        print(f"✓ {split_name} already exists, skipping")
        return

    encoder.eval()
    emb_file   = open(SAVE_PATH + f'{split_name}_embeddings_tmp.bin', 'wb')
    label_file = open(SAVE_PATH + f'{split_name}_labels_tmp.bin',     'wb')
    names_list = []
    total      = 0

    with torch.no_grad():
        for waveforms, labels, names in tqdm(loader, desc=f"Extracting {split_name}"):
            waveforms = waveforms.squeeze(1).to(device)
            with torch.cuda.amp.autocast():
                emb = encoder(waveforms).last_hidden_state
            emb_file.write(emb.cpu().float().numpy().tobytes())
            label_file.write(labels.numpy().tobytes())
            names_list.extend(names)
            total += len(waveforms)
            del emb, waveforms
            torch.cuda.empty_cache()

    emb_file.close()
    label_file.close()

    print(f"Converting {split_name} to npy...")
    emb_data   = np.frombuffer(
        open(SAVE_PATH + f'{split_name}_embeddings_tmp.bin', 'rb').read(),
        dtype=np.float32
    ).reshape(total, 149, 768)
    label_data = np.frombuffer(
        open(SAVE_PATH + f'{split_name}_labels_tmp.bin', 'rb').read(),
        dtype=np.float32
    ).reshape(total, 5)

    np.save(SAVE_PATH + f'{split_name}_embeddings.npy', emb_data)
    np.save(SAVE_PATH + f'{split_name}_labels.npy',     label_data)
    np.save(SAVE_PATH + f'{split_name}_names.npy',      np.array(names_list))

    os.remove(SAVE_PATH + f'{split_name}_embeddings_tmp.bin')
    os.remove(SAVE_PATH + f'{split_name}_labels_tmp.bin')
    print(f"✓ Saved {split_name}: ({total}, 149, 768)")
    del emb_data, label_data
    gc.collect()

precompute_embeddings(train_loader_raw, 'train')

In [ ]:
def precompute_embeddings(loader, split_name, encoder, is_whisper=False):
    save_file = SAVE_PATH + f'{split_name}_embeddings.npy'
    if os.path.exists(save_file):
        print(f"✓ {split_name} already exists, skipping")
        return

    encoder.eval()
    names_list  = []
    labels_list = []
    emb_list    = []
    total       = 0
    chunk_idx   = 0
    CHUNK_SIZE  = 50  # save every 50 batches

    with torch.no_grad():
        for waveforms, labels, names in tqdm(loader, desc=f"Extracting {split_name}"):
            waveforms = waveforms.to(device)

            with torch.amp.autocast('cuda'):
                if is_whisper:
                    emb = encoder.encoder(waveforms).last_hidden_state
                    emb = emb.mean(dim=1, keepdim=True)  # ← pool: (B, 1500, 1280) → (B, 1, 1280)
                else:
                    emb = encoder(waveforms.squeeze(1)).last_hidden_state

            emb_list.append(emb.cpu().float().numpy())
            labels_list.append(labels.numpy())
            names_list.extend(names)
            total += len(waveforms)

            # Save chunk every 50 batches
            if len(emb_list) >= CHUNK_SIZE:
                chunk = np.concatenate(emb_list, axis=0)
                np.save(SAVE_PATH + f'{split_name}_chunk_{chunk_idx}.npy', chunk)
                chunk_idx += 1
                del chunk, emb_list
                emb_list = []
                gc.collect()

            del emb, waveforms
            torch.cuda.empty_cache()

    # Save remaining
    if emb_list:
        chunk = np.concatenate(emb_list, axis=0)
        np.save(SAVE_PATH + f'{split_name}_chunk_{chunk_idx}.npy', chunk)
        chunk_idx += 1
        del chunk, emb_list
        gc.collect()

    np.save(SAVE_PATH + f'{split_name}_labels.npy', np.concatenate(labels_list))
    np.save(SAVE_PATH + f'{split_name}_names.npy',  np.array(names_list))

    print(f"Saved {chunk_idx} chunks — merging...")

    first_chunk = np.load(SAVE_PATH + f'{split_name}_chunk_0.npy')
    emb_shape   = (total, first_chunk.shape[1], first_chunk.shape[2])
    del first_chunk

    final = np.lib.format.open_memmap(
        SAVE_PATH + f'{split_name}_embeddings.npy',
        mode='w+',
        dtype=np.float32,
        shape=emb_shape
    )

    idx = 0
    for i in range(chunk_idx):
        chunk = np.load(SAVE_PATH + f'{split_name}_chunk_{i}.npy')
        final[idx:idx+len(chunk)] = chunk
        idx += len(chunk)
        del chunk
        os.remove(SAVE_PATH + f'{split_name}_chunk_{i}.npy')
        gc.collect()

    del final
    print(f"✓ Saved {split_name}: {emb_shape}")
    gc.collect()

In [ ]:
precompute_embeddings(test_loader_raw, 'test')

del encoder
torch.cuda.empty_cache()
gc.collect()
print("✓ Encoder freed")

In [ ]:
import shutil

# Copy from Drive to local Colab disk (much faster reads)
LOCAL_PATH = '/content/embeddings/'
os.makedirs(LOCAL_PATH, exist_ok=True)

print("Copying embeddings to local disk...")
for split in ['train', 'val', 'test']:
    for f in ['embeddings', 'labels', 'names']:
        src = SAVE_PATH + f'{split}_{f}.npy'
        dst = LOCAL_PATH + f'{split}_{f}.npy'
        if not os.path.exists(dst):
            print(f"  Copying {split}_{f}.npy...")
            shutil.copy(src, dst)
        else:
            print(f"  ✓ {split}_{f}.npy already local")
print("✓ All embeddings local")

In [ ]:
encoder_name = 'wav2vec'
model_path   = 'facebook/wav2vec2-base'

train_loader_raw = make_raw_loader(train_df_valid)
val_loader_raw   = make_raw_loader(val_df_valid)
test_loader_raw  = make_raw_loader(test_df_valid)

encoder = AutoModel.from_pretrained(model_path).to(device)
encoder.eval()

precompute_embeddings(train_loader_raw, f'train_{encoder_name}', encoder, is_whisper=False)
precompute_embeddings(val_loader_raw,   f'val_{encoder_name}',   encoder, is_whisper=False)
precompute_embeddings(test_loader_raw,  f'test_{encoder_name}',  encoder, is_whisper=False)

del encoder
torch.cuda.empty_cache()
gc.collect()
print(f"✓ {encoder_name} done")

In [ ]:
encoder_name = 'voc2vec'
model_path   = 'alkiskoudounas/voc2vec-hubert-ls-pt'

train_loader_raw = make_raw_loader(train_df_valid)
val_loader_raw   = make_raw_loader(val_df_valid)
test_loader_raw  = make_raw_loader(test_df_valid)

encoder = AutoModel.from_pretrained(model_path).to(device)
encoder.eval()

precompute_embeddings(train_loader_raw, f'train_{encoder_name}', encoder, is_whisper=False)
precompute_embeddings(val_loader_raw,   f'val_{encoder_name}',   encoder, is_whisper=False)
precompute_embeddings(test_loader_raw,  f'test_{encoder_name}',  encoder, is_whisper=False)

del encoder
torch.cuda.empty_cache()
gc.collect()
print(f"✓ {encoder_name} done")

In [ ]:
encoder_name = 'whisper'
model_path   = 'openai/whisper-large-v3-turbo'

train_loader_raw = make_raw_loader(train_df_valid, is_whisper=True)
val_loader_raw   = make_raw_loader(val_df_valid,   is_whisper=True)
test_loader_raw  = make_raw_loader(test_df_valid,  is_whisper=True)

encoder = AutoModel.from_pretrained(model_path).to(device)
encoder.eval()

precompute_embeddings(train_loader_raw, f'train_{encoder_name}', encoder, is_whisper=True)
precompute_embeddings(val_loader_raw,   f'val_{encoder_name}',   encoder, is_whisper=True)
precompute_embeddings(test_loader_raw,  f'test_{encoder_name}',  encoder, is_whisper=True)

del encoder
torch.cuda.empty_cache()
gc.collect()
print(f"✓ {encoder_name} done")

In [ ]:
for encoder_name in ENCODERS.keys():
    for split in ['train', 'val', 'test']:
        f = SAVE_PATH + f'{split}_{encoder_name}_embeddings.npy'
        if os.path.exists(f):
            size = os.path.getsize(f) / 1e9
            print(f"{split}_{encoder_name}: {size:.2f} GB")
        else:
            print(f"{split}_{encoder_name}: NOT FOUND")

In [ ]:
for encoder_name in ENCODERS.keys():
    for split in ['train', 'val', 'test']:
        for f in ['embeddings', 'labels', 'names']:
            src = SAVE_PATH + f'{split}_{encoder_name}_{f}.npy'
            dst = LOCAL_PATH + f'{split}_{encoder_name}_{f}.npy'

            size = os.path.getsize(src) / 1e9
            if size > 10:  # skip files larger than 10GB
                print(f"Skipping {split}_{encoder_name}_{f}.npy ({size:.1f}GB) — too large")
                continue

            if not os.path.exists(dst):
                print(f"  Copying {split}_{encoder_name}_{f}.npy ({size:.1f}GB)...")
                shutil.copy(src, dst)
            else:
                print(f"  ✓ {split}_{encoder_name}_{f}.npy already local")

print("✓ Done copying")

In [ ]:
import shutil

# Copy from Drive to local Colab disk (much faster reads)
LOCAL_PATH = '/content/embeddings/'
os.makedirs(LOCAL_PATH, exist_ok=True)

print("Copying embeddings to local disk...")
for encoder_name in ENCODERS.keys():
    for split in ['train', 'val', 'test']:
        for f in ['embeddings', 'labels', 'names']:
            src = SAVE_PATH + f'{split}_{encoder_name}_{f}.npy'
            dst = LOCAL_PATH + f'{split}_{encoder_name}_{f}.npy'
            if not os.path.exists(dst):
                print(f"  Copying {split}_{encoder_name}_{f}.npy...")
                shutil.copy(src, dst)
            else:
                print(f"  ✓ {split}_{encoder_name}_{f}.npy already local")
print("✓ All embeddings local")

In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, split_name, in_ram=False):
        print(f"Loading {split_name}...")
        if in_ram:
            self.embeddings = torch.from_numpy(
                np.load(LOCAL_PATH + f'{split_name}_embeddings.npy')
            ).float()
            self.labels = torch.from_numpy(
                np.load(LOCAL_PATH + f'{split_name}_labels.npy')
            ).float()
            self.in_ram = True
        else:
            self.embeddings = np.load(LOCAL_PATH + f'{split_name}_embeddings.npy', mmap_mode='r')
            self.labels     = np.load(LOCAL_PATH + f'{split_name}_labels.npy',     mmap_mode='r')
            self.in_ram     = False
        self.names = np.load(LOCAL_PATH + f'{split_name}_names.npy', mmap_mode='r')

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        if self.in_ram:
            return self.embeddings[idx], self.labels[idx], str(self.names[idx])
        else:
            return (
                torch.tensor(self.embeddings[idx].copy(), dtype=torch.float32),
                torch.tensor(self.labels[idx].copy(),     dtype=torch.float32),
                str(self.names[idx])
            )

def make_embedding_loader(split_name, shuffle=False, in_ram=False):
    return DataLoader(
        EmbeddingDataset(split_name, in_ram=in_ram),
        batch_size=batch_size,
        num_workers=0,
        pin_memory=False,
        shuffle=shuffle
    )

# Usage per encoder:
train_loader = make_embedding_loader(f'train_{encoder_name}', shuffle=True, in_ram=True)
val_loader   = make_embedding_loader(f'val_{encoder_name}',                 in_ram=True)
test_loader  = make_embedding_loader(f'test_{encoder_name}',                in_ram=False)

In [ ]:
import torch
import torch.nn as nn
import math

class AttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, x):
        scores  = self.attention(x)
        weights = torch.softmax(scores, dim=1)
        pooled  = torch.sum(x * weights, dim=1)
        return pooled


class StutterNet(nn.Module):
    def __init__(self,input_dim=768):
        super().__init__()

        # Layer 1 — CNN
        self.cnn = nn.Sequential(
            nn.Conv1d(
                in_channels=input_dim,
                out_channels=50,
                kernel_size=2,
                padding=0),
            nn.ReLU(),
        )

        # Layer 2 — Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=50,
            nhead=10,
            dim_feedforward=256,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # Layer 3 — Attention Pooling
        self.attention = AttentionPooling(hidden_size=50)

        # Layer 4 — Dense
        self.dense = nn.Sequential(
            nn.Linear(50, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # Layer 5 — 5 Output Heads
        self.head_prolongation = nn.Linear(128, 1)
        self.head_block        = nn.Linear(128, 1)
        self.head_soundrep     = nn.Linear(128, 1)
        self.head_wordrep      = nn.Linear(128, 1)
        self.head_interjection = nn.Linear(128, 1)

    def forward(self, x):
        # x: (B, T, 768)

        # CNN
        x = x.permute(0, 2, 1)          # (B, 768, T)
        x = self.cnn(x)                  # (B, 50, T-1)
        x = x.permute(0, 2, 1)          # (B, T-1, 50)

        # Entropy invariance scaling (Liu et al.)
        seq_len = x.size(1)
        scale   = math.log(seq_len) if seq_len > 1 else 1.0
        x = x * scale

        # Transformer
        x = self.transformer(x)          # (B, T-1, 50)

        # Attention Pooling
        x = self.attention(x)            # (B, 50)

        # Dense
        x = self.dense(x)               # (B, 128)

        return {
            'prolongation': self.head_prolongation(x).squeeze(-1),
            'block':        self.head_block(x).squeeze(-1),
            'soundrep':     self.head_soundrep(x).squeeze(-1),
            'wordrep':      self.head_wordrep(x).squeeze(-1),
            'interjection': self.head_interjection(x).squeeze(-1)
        }

model = StutterNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params/1e6:.2f}M")
print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
from torch.utils.tensorboard import SummaryWriter

classes = ['prolongation', 'block', 'soundrep', 'wordrep', 'interjection']

def train(epoch):
    model.train()
    running_loss = {h: 0 for h in classes}
    correct      = {h: 0 for h in classes}
    total        = {h: 0 for h in classes}

    for data in tqdm(train_loader, desc="Training"):
        inputs  = data[0].to(device, non_blocking=True)
        targets = {h: data[1][:, i].to(device, non_blocking=True)
                   for i, h in enumerate(classes)}

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            loss = 0
            for h in classes:
                head_loss = criterion(outputs[h], targets[h])
                running_loss[h] += head_loss.item()
                loss += head_loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        for h in classes:
            predicted = torch.round(torch.sigmoid(outputs[h]))
            total[h]   += targets[h].size(0)
            correct[h] += predicted.eq(targets[h]).sum().item()

    for h in classes:
        writer.add_scalar(f"{h} Loss/train",     running_loss[h] / len(train_loader), epoch)
        writer.add_scalar(f"{h} Accuracy/train", 100. * correct[h] / total[h],        epoch)


def eval(epoch):
    model.eval()
    running_loss = {h: 0 for h in classes}
    correct      = {h: 0 for h in classes}
    total        = {h: 0 for h in classes}

    with torch.no_grad():
        for data in tqdm(val_loader, desc="Validating"):
            inputs  = data[0].to(device, non_blocking=True)
            targets = {h: data[1][:, i].to(device, non_blocking=True)
                       for i, h in enumerate(classes)}

            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                for h in classes:
                    head_loss = criterion(outputs[h], targets[h])
                    running_loss[h] += head_loss.item()

            for h in classes:
                predicted = torch.round(torch.sigmoid(outputs[h]))
                total[h]   += targets[h].size(0)
                correct[h] += predicted.eq(targets[h]).sum().item()

    avg_val_loss = sum(running_loss.values()) / (len(val_loader) * len(classes))

    for h in classes:
        writer.add_scalar(f"{h} Loss/val",     running_loss[h] / len(val_loader), epoch)
        writer.add_scalar(f"{h} Accuracy/val", 100. * correct[h] / total[h],       epoch)

    return avg_val_loss

In [ ]:
results = {}

for encoder_name in ENCODERS.keys():
    print(f"\n{'='*50}")
    print(f"Training with: {encoder_name}")
    print(f"{'='*50}")

    # Load embeddings for this encoder
    train_loader = make_embedding_loader(f'train_{encoder_name}', shuffle=True, in_ram=True)
    val_loader   = make_embedding_loader(f'val_{encoder_name}',                 in_ram=True)
    test_loader  = make_embedding_loader(f'test_{encoder_name}',                in_ram=False)

    #  correct model per encoder
    if encoder_name == 'whisper':
        model = StutterNet(input_dim=1280).to(device)
    else:
        model = StutterNet(input_dim=768).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.BCEWithLogitsLoss()
    scaler    = GradScaler(device=device)
    writer    = SummaryWriter(comment=f"_{encoder_name}")

    best_val_loss = float('inf')

    for epoch in tqdm(range(1, num_epochs + 1), desc=f"Epoch [{encoder_name}]"):
        train(epoch)
        eval_loss = eval(epoch)

        if eval_loss < best_val_loss:
            best_val_loss = eval_loss
            torch.save({
                'epoch':      epoch,
                'model':      model.state_dict(),
                'optimizer':  optimizer.state_dict(),
                'val_loss':   best_val_loss,
                'encoder':    encoder_name,
                'input_dim':  1280 if encoder_name == 'whisper' else 768
            }, f'/content/drive/MyDrive/stutter_embeddings/best_{encoder_name}.pth')
            print(f"  ✓ Saved best {encoder_name} (val_loss: {best_val_loss:.4f})")

        torch.cuda.empty_cache()

    writer.close()
    results[encoder_name] = best_val_loss
    print(f"✓ {encoder_name} done — val_loss: {best_val_loss:.4f}")

# Compare
print("\n" + "="*50)
print("RESULTS COMPARISON:")
print("="*50)
for encoder_name, val_loss in results.items():
    print(f"{encoder_name:10} - val_loss: {val_loss:.4f}")

In [ ]:
# Load best model for each encoder and test
for encoder_name in ENCODERS.keys():
    print(f"\n{'='*50}")
    print(f"Testing: {encoder_name}")
    print(f"{'='*50}")

    # Load best model
    checkpoint = torch.load(
        f'/content/drive/MyDrive/stutter_embeddings/best_{encoder_name}.pth'
    )
    model = StutterNet(input_dim=checkpoint['input_dim']).to(device)
    model.load_state_dict(checkpoint['model'])
    model.eval()

    test_loader = make_embedding_loader(f'test_{encoder_name}', in_ram=False)

    total             = {h: 0 for h in classes}
    correct           = {h: 0 for h in classes}
    running_loss      = {h: 0 for h in classes}
    predicted_stutter = {h: 0 for h in classes}
    labels_stutter    = {h: 0 for h in classes}
    correct_stutter   = {h: 0 for h in classes}

    with torch.no_grad():
        for data in tqdm(test_loader, desc="Testing"):
            inputs  = data[0].to(device, non_blocking=True)
            targets = {h: data[1][:, i].to(device, non_blocking=True)
                       for i, h in enumerate(classes)}

            with torch.amp.autocast('cuda'):
                outputs = model(inputs)

            for h in classes:
                head_loss        = criterion(outputs[h], targets[h])
                running_loss[h] += head_loss.item()
                probs            = torch.sigmoid(outputs[h])
                predicted_labels = torch.round(probs)
                total[h]        += targets[h].size(0)
                correct[h]      += predicted_labels.eq(targets[h]).sum().item()
                predicted_stutter[h] += predicted_labels.sum().item()
                labels_stutter[h]    += targets[h].sum().item()
                correct_stutter[h]   += (predicted_labels * targets[h]).sum().item()

    print(f"\n{encoder_name} Results:")
    print("="*40)
    for h in classes:
        accu      = 100. * correct[h] / total[h]
        precision = correct_stutter[h] / predicted_stutter[h] if predicted_stutter[h] > 0 else 0
        recall    = correct_stutter[h] / labels_stutter[h]    if labels_stutter[h]    > 0 else 0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        print(f"\n  {h}")
        print(f"  Accuracy  : {accu:.2f}%")
        print(f"  Precision : {precision:.4f}")
        print(f"  Recall    : {recall:.4f}")
        print(f"  F1 Score  : {f1:.4f}")

In [ ]:
exportColumns['Block']